<img src="Screenshot 2026-08-26 at 7.15.43 PM.png">
<img src="Screenshot 2026-08-26 at 7.06.28 PM.png">
<img src="Screenshot 2026-08-26 at 7.08.39 PM.png">
<img src="Screenshot 2026-08-26 at 7.09.02 PM.png">

## PART 3: DATA PREPROCESSING
#### Step 1: Load Dataset

In [30]:
import pandas as pd

# Load CSV file
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


##### just for understanding

In [6]:
text="What is the cApital of France?"
print(text.lower())
print(text.replace("?",""))
print(text.split())

what is the capital of france?
What is the cApital of France
['What', 'is', 'the', 'cApital', 'of', 'France?']


### Step 2: Tokenization
Tokenization means splitting text into individual words.

In [ ]:
def tokenize(text):
 # Convert every letter in text to lowercase
 text = text.lower() 
 # Remove question marks and inverted commas
 text = text.replace('?', '') #replace ? with nothing
 text = text.replace(',', '')
 # Split by spaces # return list of words
 return text.split()

# Example
sentence = "What is the capital of France?"
tokens = tokenize(sentence)
print(tokens)

['what', 'is', 'the', 'capital', 'of', 'france']


### Step 3: Build Vocabulary
Vocabulary is a dictionary mapping each unique word to a unique index

vocab -> { word1:1 ,word2 : 2, word3: 3 ,   ...}

In [32]:
# Start with UNKNOWN token (index 0)
vocab = {'<UNK>':0}

In [ ]:
def build_vocab(row): # sends row by row

  # Tokenize both question and answer
  tokenized_question = tokenize(row['question']) #list of words of that row in question feature
  tokenized_answer = tokenize(row['answer']) #list of words of that row in answer feature

  # Merge both lists
  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:
    # add new words to vocabulary
    if token not in vocab:
      vocab[token] = len(vocab)

  return vocab


In [34]:
df.apply(build_vocab, axis=1)

,0
0,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
1,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
2,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
3,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
4,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
...,...
85,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
86,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
87,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
88,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."


#### df has not changed though (no inplace=True done),we are only updating vocab dictionary

In [35]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 "'to": 12,
 'kill': 13,
 'a': 14,
 "mockingbird'": 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 "'1984'": 67,
 'george-orwell': 68,
 'currency': 69,
 '

In [36]:
len(vocab)

326

### Step 4: Convert Text to Numerical Indices from our existing vocab

In [ ]:
# convert words to numerical indices during prediction from our existing vocabulary
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>']) # use UNK token

  return indexed_text

In [38]:
question="What is campusx"
indices=text_to_indices(question, vocab)
print(f"Question: {question}")
print(f"Indices: {indices}")

Question: What is campusx
Indices: [1, 2, 0]


### PART 4: CUSTOM DATASET AND DATALOADER

In [39]:
import torch
from torch.utils.data import Dataset, DataLoader

In [40]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    # gets question and answer as text
    question=self.df.iloc[index]['question']
    answer=self.df.iloc[index]["answer"]
    numerical_question = text_to_indices(question, self.vocab)
    numerical_answer = text_to_indices(answer, self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

### Create dataset and dataloader

In [41]:
dataset = QADataset(df, vocab)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Test loading
question_tensor,answer_tensor=dataset[0]
print(f"Question indices: {question_tensor}")
print(f"Answer indices: {answer_tensor}")

Question indices: tensor([1, 2, 3, 4, 5, 6])
Answer indices: tensor([7])


In [42]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[ 42, 137,   2, 227, 143,   3, 228, 229]]) tensor([156])
tensor([[ 78,  79, 290,  81,  19,  14, 291]]) tensor([85])
tensor([[ 42, 252, 253, 118, 254, 255]]) tensor([256])
tensor([[ 10,  11, 190, 159, 191]]) tensor([192])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([36])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([116])
tensor([[ 10,  29, 130, 131]]) tensor([132])
tensor([[ 42,   2,   3, 211, 137, 169, 212, 170]]) tensor([113])
tensor([[ 42, 137,   2,  62,  39,   3, 324, 325]]) tensor([6])
tensor([[  1,   2,   3, 181, 182, 183, 184]]) tensor([185])
tensor([[ 42,  86,  87, 243, 244,  19,  39, 245]]) tensor([246])
tensor([[  1,   2,   3, 222,   5, 223, 224, 225]]) tensor([226])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[  1,   2,   3,  33,  34,   5, 247]]) tensor([248])
tensor([[ 42, 292, 293, 118, 294, 159, 295, 296]]) tensor([297])
tensor([[  1,   2,   3,   4,   5, 288]]) tensor([289])
tensor([

## PART 5:RNN MODEL ARCHITECTURE

<img src="Screenshot 2026-08-26 at 7.44.26 PM.png">

No, you **cannot** use `nn.Sequential` here directly, and here's why:

## The Problem

`nn.Sequential` only works when:
1. Each layer takes **exactly one input** and produces **exactly one output**
2. Data flows linearly from one layer to the next

But `nn.RNN` returns **two things**: `(output, hidden_state)`, while your next layer `nn.Linear` only expects the hidden state.



In [43]:
import torch.nn as nn

In [51]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()

    #Embedding layer: converts word indices,each word to vectors of 50 dim embedding
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)

    #RNN layer:processes sequence with memory
    #batch_first=True means input shape is (batch,seq_len,features)
    self.rnn = nn.RNN(50, 64, batch_first=True)

    # Fully connected layer:maps to vocabulary size
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [52]:
## for understanding of embedding,each word to vectors of 50 embedding dim
print(dataset[0])
print(dataset[0][0])

x=nn.Embedding(len(vocab),embedding_dim=50)
x(dataset[0][0]).shape

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))
tensor([1, 2, 3, 4, 5, 6])


torch.Size([6, 50])

## PART6 : TRAINING THE MODEL

In [53]:
# hyperparameters
learning_rate = 0.001
epochs = 20

# Initialize model
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleRNN(len(vocab))
model.to(device)

#Loss function for multiclass classification
criterion = nn.CrossEntropyLoss()

#Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## Training Loop


In [55]:
for epoch in range(epochs):
  model.train()
  total_loss=0

  for question, answer in dataloader:
    question = question.to(device)
    answer = answer.to(device)

    #clear gradients
    optimizer.zero_grad()

    #Forward pass
    predictions=model(question) #Shape:(1,vocab_size)

    #Calculate loss
    #answer is tensor([index]),need to ensure its the right shape

    loss=criterion(predictions,answer[0])

    #Backward pass
    loss.backward()

    #Update weights
    optimizer.step()

    #Accumulate loss
    total_loss+=loss.item()

  print(f"Epoch {epoch+1}/{epochs},loss:{total_loss:.4f}")

Epoch 1/20,loss:522.5291
Epoch 2/20,loss:456.2611
Epoch 3/20,loss:380.1803
Epoch 4/20,loss:316.0171
Epoch 5/20,loss:263.1076
Epoch 6/20,loss:214.5475
Epoch 7/20,loss:169.6770
Epoch 8/20,loss:132.0185
Epoch 9/20,loss:101.8471
Epoch 10/20,loss:78.7520
Epoch 11/20,loss:61.4596
Epoch 12/20,loss:48.4362
Epoch 13/20,loss:39.0174
Epoch 14/20,loss:31.9121
Epoch 15/20,loss:26.4530
Epoch 16/20,loss:21.9738
Epoch 17/20,loss:18.6027
Epoch 18/20,loss:15.9170
Epoch 19/20,loss:13.8697
Epoch 20/20,loss:12.0188


## PART 7 : MAKING PREDICTIONS

In [56]:
import torch.nn.functional

In [ ]:
def predict(model, question, threshold=0.5):

  # step 1: convert question to numbers/indices
  numerical_question = text_to_indices(question, vocab)

  # step2:convert to tensor and add batch dimension
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # step 3:Get model predictions(logits)
  output = model(question_tensor)

  # step 4: convert logits to probabilities using softmax
  probs = torch.nn.functional.softmax(output, dim=1)

  # step5 : find max probability and its  index
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [ ]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [ ]:
list(vocab.keys())[7]

'paris'